In [0]:
print("lee")

In [0]:
spark.conf.set("spark.sql.shuffle.partitions",50)

In [0]:
df=spark.read.format("csv").option("header","true").load("/Volumes/external-catalog/default/test-volume/Emp_Table.csv")
df.show()
#df.write.format("parquet").mode("overwrite").save(")

In [0]:
display(df)

In [0]:
emp_sal_less_than_thousand=df.filter(df['SAL']<1000)
display(emp_sal_less_than_thousand)
selected_columns=['ENAME','JOB','SAL','MGR']

emp_sal_less_than_thousand_df=df.select(*[c for c in selected_columns if c in df.columns])

In [0]:
emp_sal_less_than_thousand_df.write.format("delta").mode("overwrite").saveAsTable("`external-catalog`.default.low_salaried_employees")

In [0]:
#list down delta table versions
history_df=spark.sql("DESCRIBE HISTORY `external-catalog`.default.low_salaried_employees")
display(history_df)

In [0]:
#read delata table
df_lowsalryEmp=spark.read.format("delta").table("`external-catalog`.default.low_salaried_employees")
display(df_lowsalryEmp)


In [0]:
#show schema of delta table  
df_lowsalryEmp.printSchema()

In [0]:
from pyspark.sql import Row


In [0]:
dummy_data=[Row(ENAME='LEE',JOB='CLERK',SAL='100',MGR='7902')]

In [0]:
dummy_df=spark.createDataFrame(dummy_data)
#append dummy data
dummy_df.write.format("delta").mode("append").saveAsTable("`external-catalog`.default.low_salaried_employees")
display(dummy_df)

In [0]:
#read delata table
df_lowsalryEmp=spark.read.format("delta").table("`external-catalog`.default.low_salaried_employees")
display(df_lowsalryEmp)

In [0]:
#list down delta table versions
history_df=spark.sql("DESCRIBE HISTORY `external-catalog`.default.low_salaried_employees")
display(history_df)

In [0]:
#list version 0 of delat table
df_lowsalryEmp=spark.read.format("delta").option("versionAsOf",0).table("`external-catalog`.default.low_salaried_employees")
display(df_lowsalryEmp)

In [0]:

deleet_df=spark.read.format("delta")\
    .option("timestampAsOf","2026-07-06T19:50:50.793+00:00")\
    .table("`external-catalog`.default.low_salaried_employees")
display(deleet_df)


In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data")

In [0]:
from pyspark.sql.functions import col,lower
#example logical transformation:lowercase JOB for all emplyees
transformed_df=df.withColumn("JOB",lower(col("JOB")))
#write to delta table
outputpath="/Volumes/external-catalog/default/employee_transformed_data"
transformed_df.write.mode("overwrite").partitionBy("job").format("delta").save(outputpath)

In [0]:
spark.sql("CREATE VOLUME IF NOT EXISTS `external-catalog`.default.employee_transformed_data_parquet")

In [0]:
transformed_df1=df.withColumn("JOB",lower(col("JOB")))
#write to delta table
outputpath="/Volumes/external-catalog/default/employee_transformed_data_parquet"
transformed_df1.write.mode("overwrite").partitionBy("job").format("parquet").save(outputpath)